In [ ]:
import math
import time
from dataclasses import dataclass

import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

In [ ]:
@dataclass(frozen=True)
class ShapeSpec:
    b: int
    c: int
    h: int
    w: int


SIZES  = (2, 4, 8, 16, 32, 64)
SHAPES = [ShapeSpec(b=b, c=c, h=hw, w=hw) for b in SIZES for c in SIZES for hw in SIZES]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE  = torch.float16 if (DEVICE.type == "cuda") else torch.float32

print(f"[device]: {DEVICE}")

In [ ]:
def pearsonr(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    x = x.flatten(1)
    y = y.flatten(1)
    x = x - x.mean(dim=-1, keepdim=True)
    y = y - y.mean(dim=-1, keepdim=True)
    num = (x * y).sum(dim=-1)
    den = x.norm(dim=-1) * y.norm(dim=-1)
    return num / den.clamp_min(1e-8)

In [ ]:
def _offsets(kernel_radius: int) -> torch.Tensor:
    r = int(kernel_radius)
    dh = torch.arange(-r, r + 1, device=DEVICE)
    dw = torch.arange(-r, r + 1, device=DEVICE)
    grid_dh, grid_dw = torch.meshgrid(dh, dw, indexing="ij")
    offsets = torch.stack([grid_dh.flatten(), grid_dw.flatten()], dim=-1)
    offsets = offsets[(offsets[:, 0] != 0) | (offsets[:, 1] != 0)]
    return offsets


def max_fn(fm: torch.Tensor, kernel_radius: int) -> torch.Tensor:
    b, c, h, w = fm.shape
    offsets = _offsets(kernel_radius)
    best = fm.new_full((), -float("inf"))
    for c1 in range(c):
        x0 = fm[:, c1]
        for c2 in range(c1 + 1, c):
            y0 = fm[:, c2]
            for dh, dw in offsets.tolist():
                dh = int(dh)
                dw = int(dw)
                hs = max(0, -dh)
                he = min(h, h - dh)
                ws = max(0, -dw)
                we = min(w, w - dw)
                if (he <= hs) or (we <= ws):
                    continue
                x = x0[:, hs:he, ws:we]
                y = y0[:, hs + dh : he + dh, ws + dw : we + dw]
                sim = pearsonr(x, y).max()
                best = torch.maximum(best, sim)
    return best

In [ ]:
def luca_fn(fm: torch.Tensor, kernel_radius: int) -> torch.Tensor:
    b, c, h, w = fm.shape
    r          = int(kernel_radius)
    k          = 2 * r + 1
    k2         = k * k
    patches = F.unfold(fm, kernel_size=k, dilation=1, padding=r, stride=1)  # (b, c * k2, l)

    l       = patches.shape[-1]
    patches = patches.view(b, c, k, k, l)
    patches = patches.flatten(2, 3)        # (b, c, k2,      l)
    patches = patches[:, :, (k2 // 2):, :] # (b, c, (k2 // 2), l)

    center = patches[:, :, 0:1, :] # (b, c, 1,             l)
    neigh  = patches[:, :, 1:,  :] # (b, c, (k2 // 2) - 1, l)

    bl = b * h * w
    n_off = neigh.shape[2]
    v = center.permute(1, 0, 2, 3).reshape(c, bl)
    n = neigh.permute(1, 2, 0, 3).reshape(c, n_off, bl)
    v_exp = v.unsqueeze(1).unsqueeze(1).expand(c, c, n_off, bl)
    n_exp = n.unsqueeze(0).expand(c, c, n_off, bl)
    x_flat = v_exp.contiguous().reshape(-1, bl)
    y_flat = n_exp.contiguous().reshape(-1, bl)
    sims_flat = pearsonr(x_flat, y_flat)
    sims = sims_flat.reshape(c, c, n_off)
    sims = sims.max(dim=-1).values
    mask = torch.triu(torch.ones((c, c), device=DEVICE, dtype=torch.bool), diagonal=1)
    sims = sims.masked_fill(~mask, -float("inf"))
    return sims.max()

In [ ]:
def _sync_if_cuda() -> None:
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()


def time_fn(fn, fm: torch.Tensor, kernel_radius: int, warmup: int = 2, runs: int = 10) -> float:
    for _ in range(warmup):
        _ = fn(fm, kernel_radius)
    _sync_if_cuda()
    times: list[float] = []
    for _ in range(runs):
        t0 = time.perf_counter()
        _ = fn(fm, kernel_radius)
        _sync_if_cuda()
        t1 = time.perf_counter()
        times.append(t1 - t0)
    return float(sum(times) / max(1, len(times)))


def make_fm(shape: ShapeSpec) -> torch.Tensor:
    return torch.randn((shape.b, shape.c, shape.h, shape.w), device=DEVICE, dtype=DTYPE)


def benchmark(
    shapes: list[ShapeSpec],
    kernel_radius: int = 2,
    warmup: int = 2,
    runs: int = 10,
) -> list[dict]:
    fns = {
        "max_fn": max_fn, 
        "luca_fn": luca_fn
    }

    results: list[dict] = []
    for shape in tqdm(shapes):
        fm = make_fm(shape)
        row = {
            "b": shape.b, 
            "c": shape.c, 
            "h": shape.h, 
            "w": shape.w
        }
        for name, fn in fns.items():
            row[name] = time_fn(fn, fm, kernel_radius=kernel_radius, warmup=warmup, runs=runs)
        results.append(row)
    return results

In [ ]:
KERNEL_RADIUS = 2
WARMUP = 1
RUNS = 10

results = benchmark(SHAPES, kernel_radius=KERNEL_RADIUS, warmup=WARMUP, runs=RUNS)

fastest = sorted(results, key=lambda r: min(r["max_fn"], r["luca_fn"]))[:10]
slowest = sorted(results, key=lambda r: max(r["max_fn"], r["luca_fn"]), reverse=True)[:10]

fastest, slowest[:3]